# Fourier basis — what the energy-biased training does

Supports *Fitting the amplitudes without an energy term* in Supplementary
Note 1. The recumbent fold, two ways of training each basis, everything else
as `01b`:

- **ours**: amplitudes start at zero and are updated by gradient descent with
  momentum; phases and biases by Adam (the fits of the paper);
- **Adam**: amplitudes start at zero, every parameter updated by Adam.

Both start from zero, so they differ only in how the amplitudes are updated. A
random start is not tested: gradient descent leaves unchanged whatever a
random start puts where the data cannot see, which is a standard result.

Each of `N_SEEDS` bases (the seeds of the band sweep, the first being the
main-text seed) is trained both ways, so every comparison is **paired
within a seed**: the basis is fixed and only the training changes. This is
done for two wavelength bands (`BANDS`): the main-text band, and a wider one
reaching four times shorter, where the data leave many more short modes free.
Under gradient descent the weight $\lVert\boldsymbol\omega\rVert^{-2}$ makes short
modes learn slowly; Adam rescales every amplitude to the same step, so it is
expected to use them.

Recorded every `EVERY` iterations, without interrupting the optimiser (through
the fit's `schedule` hook):

- the loss;
- $\sum_i a_i^2$, the quantity gradient descent from zero keeps small;
- the **strain energy** of the map, $\langle\sum_k(\ln\lambda_k)^2\rangle$, the mean
  squared Hencky strain, with $\lambda_k$ the principal stretches (singular values
  of $\mathbf F$, `diagnostics.strain_measures`), over a Poisson-disc sample of the
  grid (points at least 20 m apart): how much the map deforms the rock, whatever
  the parametrisation. Not square-rooted, so a ratio of two fits is a ratio of
  amounts of strain.

Adam fits much tighter than ours, so its final state is not a fair
comparison. Each Adam run is also scored **at matched loss**: the hook checks
the loss every iteration until it first falls to ours' final loss on the same
basis and band, and saves the parameters there. Adam then trains on to the end,
for the curves. Strain energy, $\sum_i a_i^2$, the scores and the amplitudes are
saved for both the matched and the final state.

At the end, also the 90th percentile of $\sum_k(\ln\lambda_k)^2$ over the points
(where the strain is concentrated), and each mode's amplitude $a_i$ and
wavelength, for the amplitude per octave.

Writes `results/energy_bias.npz`, read by `s01d_fourier_basis_figures`.
Runs in `curlew_dev`.

## Setup

In [1]:
%matplotlib inline
import json
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import torch

sys.path.insert(0, str(Path("..").resolve()))                      # publication/: diagnostics
sys.path.insert(0, str(Path("../01_recumbent_fold").resolve()))    # recumbent_inputs, synthetic_fold
import diagnostics
import recumbent_inputs

import curlew
from curlew.core import CSet, HSet
from curlew.geometry import Grid
from curlew.geology import restore
from curlew.kinematics.euler import euler_velocity
from curlew.fields.series import FSF

curlew.device = "cuda" if torch.cuda.is_available() else "cpu"
curlew.dtype = torch.float32
curlew.default_dim = 2

FOLD_DIR = Path("../01_recumbent_fold")
DATA_DIR = FOLD_DIR / "data"
P = json.loads((FOLD_DIR / "results/synthetic/fit_settings.json").read_text())   # written by 01b
RES_DIR = Path("./results")
RES_DIR.mkdir(parents=True, exist_ok=True)

## Data, grid and truth

As in `01d`.

In [2]:
section = recumbent_inputs.load_section("synthetic", DATA_DIR)
horizon_names, horizons = section["horizon_names"], section["horizons"]
gp, gv, origin = section["gp"], section["gv"], section["origin"]
anchor = torch.zeros(2, dtype=curlew.dtype, device=curlew.device)
C = CSet(gp=gp, gv=gv, eq=horizons, iq=(P["iq_samples"], [(horizons[0], horizons[1], "<")]))

G = Grid((1200, 1200), step=(5.0, 5.0), center=np.zeros(2), sampleArgs=dict(N=128))
domaxes = G.tformed_axes
grd = torch.tensor(G.coords(), dtype=curlew.dtype, device=curlew.device)
unit_true = recumbent_inputs.true_units(domaxes[0], domaxes[1], origin, DATA_DIR)
reference = recumbent_inputs.truth_reference(origin, DATA_DIR)
# Where the Hencky strain is averaged: evenly spaced grid nodes, at least STRAIN_SPACING m apart
# (the grid's Poisson-disc sample, fixed seed so every fit uses the same points).
STRAIN_SPACING = 20.0
strain_points = np.asarray(G.sample(poissonDisk=(STRAIN_SPACING, 100_000, 0)))
print(f"{len(strain_points)} strain points, at least {STRAIN_SPACING:.0f} m apart")

2580 strain points, at least 20 m apart


## Training variants

`make_ldi(variant, seed, band)` builds the LDI event of `01d` on that basis.
`modes(event)` returns the learned amplitudes $a_i$ (`C_mu` of the velocity's
Fourier field) with each mode's wavelength, from its fixed wavevector.

In [3]:
EPOCHS = 3000          # twice the paper's 2000, so the slower variants can level off
EVERY = 25

def make_ldi(variant, seed, band):
    velocity = euler_velocity(
        2, n_features=P["n_features"], n_directions=P["n_directions"], seed=int(seed),
        length_scale_range=tuple(band), stream="alpha", activation="cubic",
        energy_normalised=P.get("energy_normalised", False),
    )
    optimizer = None                                   # Adam on every parameter
    if variant != "adam":
        optimizer = {
            "amplitude": (torch.optim.SGD, dict(lr=P["lr_sgd"], momentum=P["amp_momentum"])),
            "shape": (torch.optim.Adam, dict(lr=P["lr_adam"])),
        }
    event = restore(
        f"s01_{variant}", C=C, H=HSet().zero(**P["weights"]), velocity=velocity,
        n_steps=P["n_steps"], integrator=P["integrator"], depth_axis=1,
        init_std=0.0, sigma_floor=0.0, signed_normals=True,
        learning_rate=P["lr_adam"], optimizer=optimizer, pin=anchor,
    )
    for name, h in zip(reversed(horizon_names), reversed(horizons)):
        event.addIsosurface(name, seed=h)
    return curlew.GeoModel([event]), event


def modes(event):
    # Amplitude and wavelength of every mode of the learnable Fourier field(s).
    amp, wavelength = [], []
    for m in event.field.modules():
        if isinstance(m, FSF) and hasattr(m, "C_mu"):
            k = m.fourier_projection[: m.input_dim].detach().norm(dim=0)
            amp.append(m.C_mu.detach().flatten())
            wavelength.append(2.0 * np.pi / k)
    return torch.cat(amp).cpu().numpy(), torch.cat(wavelength).cpu().numpy()


def strain_density(event):
    # Squared Hencky strain at every strain point.
    s = diagnostics.strain_measures(event, strain_points)["stretches"]
    return np.sum(np.log(s) ** 2, axis=1)


def evaluate(model, event):
    # Loss, energy, strain, scores, amplitudes and contacts of the event's current state.
    d = diagnostics.evaluate_ensemble([model], [event], grd, G.shape, domaxes[0], domaxes[1], grid=G)
    units = recumbent_inputs.match_units(d["lithology"][0], unit_true)
    s = recumbent_inputs.score_against_truth(units, domaxes[0], domaxes[1], unit_true, reference)
    with torch.no_grad():
        loss = float(event.loss().total())
        amp, wavelength = modes(event)
        density = strain_density(event)
    return dict(loss=loss, sum_a2=float(np.sum(amp ** 2)), strain=float(density.mean()),
                strain_p90=float(np.percentile(density, 90)), extrema=int(d["extrema"].sum()),
                contact_distance=np.asarray(s["contact_distance"]), iou=np.asarray(s["iou"]),
                amplitude=amp, wavelength=wavelength,
                contacts=diagnostics.ensemble_contacts(d["scalar"], domaxes[0], domaxes[1], horizons)[0])


def train(variant, seed, band, target=None):
    # With `target`, also keep the parameters at the first iteration whose loss is <= target.
    torch.manual_seed(405)
    model, event = make_ldi(variant, seed, band)
    log = dict(iteration=[], loss=[], sum_a2=[], strain=[])
    match = dict(open=target is not None)

    def record(epoch, epochs):
        logging = epoch % EVERY == 0
        if not (logging or match["open"]):
            return
        with torch.no_grad():
            loss = float(event.loss().total())
            if match["open"] and loss <= target:
                match.update(open=False, iteration=epoch,
                             state={k: v.detach().clone() for k, v in event.field.state_dict().items()})
            if logging:
                log["iteration"].append(epoch)
                log["loss"].append(loss)
                log["sum_a2"].append(float(np.sum(modes(event)[0] ** 2)))
                log["strain"].append(float(strain_density(event).mean()))

    t0 = perf_counter()
    model.prefit(epochs=EPOCHS, vb=False, early_stop=None, best=False, schedule=record)
    match["open"] = False
    record(0, 0)                                                   # the final state
    log["iteration"][-1] = EPOCHS
    final = evaluate(model, event)
    out = {k: np.asarray(v) for k, v in log.items()}
    out.update({k: v for k, v in final.items() if k not in log})
    msg = ""
    if target is not None:
        if "state" in match:
            event.field.load_state_dict(match["state"])
            event.field.pin_frame()
            out["matched"] = dict(evaluate(model, event), iteration=match["iteration"])
        else:                                                      # never reached it: score the final state
            out["matched"] = dict(final, iteration=-1)
        m = out["matched"]
        msg = (f"; at ours' loss {target:.3g}: iteration {m['iteration']}, sum a^2 {m['sum_a2']:.3g}, "
               f"strain energy {m['strain']:.3f}, IoU {np.round(m['iou'], 2)}")
    print(f"{band[0]:.0f}-{band[1]:.0f} m, seed {seed}, {variant}: {perf_counter() - t0:.0f} s, "
          f"final loss {out['loss'][-1]:.3g}, sum a^2 {out['sum_a2'][-1]:.3g}, "
          f"strain energy {out['strain'][-1]:.3f} (90th pct {out['strain_p90']:.3f}), extrema {out['extrema']}, "
          f"IoU {np.round(out['iou'], 2)}" + msg)
    return out


## Fits


In [4]:
N_SEEDS = 15
seed_rng = np.random.default_rng(P["fourier_seed"] + 91)        # as 01d and s01b
SEEDS = [int(P["fourier_seed"])]
while len(SEEDS) < N_SEEDS:
    candidate = int(seed_rng.integers(0, 2**31 - 1))
    if candidate not in SEEDS:
        SEEDS.append(candidate)

BANDS = {"main": tuple(P["lsr_ldi"]),                            # the main-text band, 1-5 km
         "wide": (250.0, P["lsr_ldi"][1])}                       # four times shorter at the short end
VARIANTS = ("ours", "adam")
runs = {}
for b, band in BANDS.items():
    runs[(b, "ours")] = [train("ours", seed, band) for seed in SEEDS]
    # Adam is also scored where its loss first reaches ours' final loss on the same basis
    runs[(b, "adam")] = [train("adam", seed, band, target=float(r["loss"][-1]))
                         for seed, r in zip(SEEDS, runs[(b, "ours")])]

1000-5000 m, seed 42, ours: 14 s, final loss 2.71e-05, sum a^2 13.9, strain energy 1.069 (90th pct 2.921), extrema 0, IoU [0.96 0.89 0.96]
1000-5000 m, seed 1123574222, ours: 13 s, final loss 8.86e-06, sum a^2 15.7, strain energy 1.119 (90th pct 3.359), extrema 0, IoU [0.92 0.83 0.95]
1000-5000 m, seed 2094121401, ours: 13 s, final loss 6.96e-05, sum a^2 15.8, strain energy 1.159 (90th pct 3.266), extrema 0, IoU [0.93 0.84 0.95]
1000-5000 m, seed 86309145, ours: 15 s, final loss 9.61e-08, sum a^2 47.7, strain energy 1.265 (90th pct 4.086), extrema 0, IoU [0.83 0.72 0.95]
1000-5000 m, seed 1099952541, ours: 15 s, final loss 3.68e-06, sum a^2 17, strain energy 1.130 (90th pct 3.309), extrema 0, IoU [0.96 0.86 0.94]
1000-5000 m, seed 505909162, ours: 13 s, final loss 5.49e-06, sum a^2 16.3, strain energy 1.107 (90th pct 3.377), extrema 0, IoU [0.93 0.85 0.96]
1000-5000 m, seed 774917502, ours: 13 s, final loss 6.44e-06, sum a^2 15.2, strain energy 1.100 (90th pct 3.256), extrema 0, IoU [0

## Save

In [5]:
arrays = {"variants": np.array(VARIANTS), "bands": np.array(list(BANDS)),
          "band_ranges": np.array(list(BANDS.values())), "seeds": np.array(SEEDS), "epochs": EPOCHS,
          "window": np.array(reference["window"]), "strain_spacing": STRAIN_SPACING}
for (b, v), rs in runs.items():
    for key in ("iteration", "loss", "sum_a2", "strain", "contact_distance", "iou", "amplitude",
                "wavelength", "strain_p90", "extrema"):
        arrays[f"{b}_{v}_{key}"] = np.stack([np.asarray(r[key]) for r in rs])     # seed x ...
        if v == "adam":                                                           # at ours' final loss
            arrays[f"{b}_adam_matched_{key}"] = np.stack([np.asarray(r["matched"][key]) for r in rs])
first = [runs[("main", v)][0] for v in VARIANTS]                                  # main-text seed
arrays.update(diagnostics.pack_contacts([r["contacts"] for r in first] + [first[1]["matched"]["contacts"]]))
np.savez(RES_DIR / "energy_bias.npz", **arrays)
print("wrote", RES_DIR / "energy_bias.npz")

# Paired, within each seed: Adam against ours, per band (fits that did not fit included here).
for b in BANDS:
    print(b, "- Adam reached ours' loss at iterations", arrays[f"{b}_adam_matched_iteration"])
    for key in ("strain", "sum_a2"):
        o = arrays[f"{b}_ours_{key}"][:, -1]
        for when, a in (("final", arrays[f"{b}_adam_{key}"][:, -1]), ("matched", arrays[f"{b}_adam_matched_{key}"])):
            print(f"  {key}, Adam {when}: Adam / ours {np.round(a / o, 2)}  (ours lower in {np.sum(a > o)}/{len(o)})")

wrote results/energy_bias.npz
main - Adam reached ours' loss at iterations [231 644 196 343 478 283 592 141 219 398 375 120 331 318  -1]
  strain, Adam final: Adam / ours [ 1.33  1.08  1.23  1.13  1.61  1.45  1.32  1.68  1.36  1.48  1.24  1.24
  1.11  1.31 16.81]  (ours lower in 15/15)
  strain, Adam matched: Adam / ours [ 1.24  1.05  1.13  1.05  1.56  1.28  1.45  1.47  1.24  1.31  1.19  0.98
  1.02  1.26 16.81]  (ours lower in 14/15)
  sum_a2, Adam final: Adam / ours [ 3.07  2.7   3.03  0.96  3.12  2.93  3.26  2.36  1.91  3.03  1.33  2.77
  2.7   3.07 31.45]  (ours lower in 14/15)
  sum_a2, Adam matched: Adam / ours [ 2.99  2.62  2.77  0.93  3.04  2.8   3.24  2.12  1.82  2.74  1.29  2.4
  2.63  2.99 31.45]  (ours lower in 14/15)
wide - Adam reached ours' loss at iterations [ 265  393  239  326  403  369 1607  335  217  349 1399  422  388  545
  289]
  strain, Adam final: Adam / ours [ 1.61  1.75  1.3   1.36  1.56  1.68  2.56  3.06 26.05  1.51  1.93  1.52
  1.47  1.67 10.19]  (ours low